In [ ]:
#Cnn+word2vec
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import re

from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    accuracy_score
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D,
    Dense, Dropout, BatchNormalization, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paths: change these to match your environment
DATA_PATH = "AmHQA.csv"
WORD2VEC_PATH = "/content/amharic_word2vec.bin"

# Load data
df = pd.read_csv(DATA_PATH, encoding="utf-8")

question_col = next(
    (c for c in ["question", "Question"] if c in df.columns), None
)
label_col = next(
    (c for c in ["category", "Category", "label", "Label"]
     if c in df.columns), None
)

if question_col is None or label_col is None:
    raise ValueError(f"Columns found: {df.columns.tolist()}")

df = df[[question_col, label_col]].copy()
df.columns = ["question", "category"]
df = df.dropna()

df["question"] = df["question"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()
df = df[
    (df["question"] != "") & (df["category"] != "")
].drop_duplicates().reset_index(drop=True)

print("Dataset size:", df.shape)
print("\nClass distribution:")
print(df["category"].value_counts())

# Load pretrained Word2Vec binary embeddings
word_vectors = KeyedVectors.load_word2vec_format(
    WORD2VEC_PATH,
    binary=True
)

print("\nWord2Vec vocabulary:", len(word_vectors))
print("Embedding dimension:", word_vectors.vector_size)
#Preprocess questions and create train, validation, and test sets
MAX_LEN = 40
MAX_WORDS = 50000

def clean_text(text):
    return re.sub(r"\s+", " ", str(text).strip())

df["question"] = df["question"].apply(clean_text)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["category"])
num_classes = len(label_encoder.classes_)

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df["question"].values,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

# Fit tokenizer on training questions only
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    filters=""
)
tokenizer.fit_on_texts(X_train_text)

def encode_questions(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(
        sequences,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

X_train = encode_questions(X_train_text)
X_val = encode_questions(X_val_text)
X_test = encode_questions(X_test_text)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
embedding_dim = word_vectors.vector_size

# Initialize embedding matrix
embedding_matrix = np.random.normal(
    0, 0.05, (vocab_size, embedding_dim)
).astype(np.float32)

embedding_matrix[0] = 0

matched_words = 0

for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue
    if word in word_vectors:
        embedding_matrix[idx] = word_vectors[word]
        matched_words += 1

print("\nClasses:", list(label_encoder.classes_))
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))
print("Matched Word2Vec words:", matched_words)
print("Embedding coverage:", round(
    100 * matched_words / max(vocab_size - 2, 1), 2
), "%")
#Build the CNN + Word2Vec model
inputs = Input(shape=(MAX_LEN,), name="question_input")

embedded = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=False,
    name="word2vec_embedding"
)(inputs)

branches = []

for kernel_size in [2, 3, 4]:
    conv = Conv1D(
        filters=128,
        kernel_size=kernel_size,
        activation="relu",
        padding="valid"
    )(embedded)

    pooled = GlobalMaxPooling1D()(conv)
    branches.append(pooled)

x = Concatenate()(branches)
x = BatchNormalization()(x)
x = Dropout(0.40)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.30)(x)

outputs = Dense(
    num_classes,
    activation="softmax",
    name="class_probabilities"
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
#train for epoch 10
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    digits=4,
    zero_division=0
))
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(num_classes)
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Confusion Matrix: CNN + Word2Vec")
plt.tight_layout()
plt.show()
# Convert true labels to one-hot/binary format
y_test_bin = label_binarize(
    y_test,
    classes=np.arange(num_classes)
)

# Calculate ROC-AUC scores
macro_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="macro"
)

weighted_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

print(f"Macro ROC-AUC:    {macro_auc:.4f}")
print(f"Weighted ROC-AUC: {weighted_auc:.4f}")

# Plot one-vs-rest ROC curves
plt.figure(figsize=(9, 7))

for i, class_name in enumerate(label_encoder.classes_):
    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    class_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{class_name} (AUC = {class_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curves: CNN + Word2Vec")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
#training loss and validation loss
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy curve
axes[0].plot(
    range(1, 11),
    history.history["accuracy"],
    marker="o",
    label="Training accuracy"
)
axes[0].plot(
    range(1, 11),
    history.history["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)
axes[0].set_title("Accuracy Across 10 Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_xticks(range(1, 11))
axes[0].legend()

# Loss curve
axes[1].plot(
    range(1, 11),
    history.history["loss"],
    marker="o",
    label="Training loss"
)
axes[1].plot(
    range(1, 11),
    history.history["val_loss"],
    marker="s",
    label="Validation loss"
)
axes[1].set_title("Loss Across 10 Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_xticks(range(1, 11))
axes[1].legend()

plt.tight_layout()
plt.show()
